# 1.摘要缓冲混合记忆示例

知识点讲解：摘要缓冲混合记忆（SummarizationMiddleware + Agent + Checkpointer）

1. 什么是摘要缓冲混合记忆（Summary Buffer Memory）
   - 把对话历史切成两部分：
       摘要区：早期对话被 LLM 压缩成一段摘要文本（长期记忆，token 恒定）
       缓冲区：近期对话保留完整原文（短期记忆，细节无损）
   - 触发机制：当历史超过阈值时，把最早的部分交给 LLM 压缩进摘要，并从缓冲区移除
   - 定位：在「信息保真度」与「token 成本」之间取得平衡，长对话场景的最佳方案

2. 三种记忆策略的对比（贯穿第 7、10、11 章）
   策略             早期信息      token 成本      额外 LLM 调用
   全量缓冲         完整保留      线性增长        无
   窗口裁剪（第10章）直接丢弃      恒定            无
   摘要缓冲（本章）  压缩保留      恒定            有（每次触发摘要都要调模型）
   - 窗口裁剪省钱但会失忆，摘要缓冲不失忆但每次压缩要额外花一次模型调用
   - 第 7 章用原生 SDK 手写过这套逻辑，本章展示 LangChain 1.x 的内置实现

3. LangChain 1.x 的实现范式（本示例核心）
   旧版由 ConversationSummaryBufferMemory 一个组件隐式完成全部工作。
   1.x 改为 Agent + Middleware + Checkpointer 三层协作：
   - create_agent    构建 Agent 执行图（基于 LangGraph）
   - Middleware      在 Agent 每步执行前后插入横切逻辑，摘要就是一种 middleware
   - Checkpointer    负责状态（含完整消息历史）的持久化，线程即会话
   好处：摘要逻辑与业务链解耦，可与其他 middleware（限流、审计、工具过滤）自由组合

4. AgentMiddleware 中间件机制
   - SummarizationMiddleware 继承自 AgentMiddleware，是官方内置中间件之一
   - 工作位置：在把消息发给模型之前拦截，检查是否触发摘要条件
   - 触发后动作：调用摘要模型压缩早期消息，用摘要消息替换原始消息序列
   - 该替换会写入 Agent 状态，因此摘要是「持久化的」，不是每次重新计算

5. SummarizationMiddleware 的核心参数
   - model              执行摘要任务的模型，可与对话模型不同
                        （常见优化：对话用强模型，摘要用便宜的小模型）
   - trigger            触发摘要的条件，支持三种单位的元组：
                          ("tokens", 300)     token 数超过 300 时触发
                          ("messages", 20)    消息条数超过 20 条时触发
                          ("fraction", 0.8)   占用上下文窗口 80% 时触发
                        也可传条件列表，任一满足即触发
   - keep               摘要后保留多少原文（缓冲区大小），同样支持三种单位：
                          ("messages", 4)     保留最近 4 条消息原文（默认 20 条）
                          ("tokens", 500)     保留最近约 500 token 的原文
   - token_counter      token 计数函数，默认 count_tokens_approximately（快速近似）
   - summary_prompt     摘要提示词模板，内置模板会引导模型输出结构化摘要
                        （含 SESSION INTENT / SUMMARY / ARTIFACTS / NEXT STEPS 分节）
   - trim_tokens_to_summarize
                        送入摘要模型的消息本身也要限长，默认 4000 token，
                        防止待摘要内容过长再次超出上下文窗口

6. trigger 与 keep 的配合关系（理解摘要节奏的关键）
   - trigger 决定「什么时候压缩」，keep 决定「压缩后留多少原文」
   - 本例 trigger=("tokens", 300)、keep=("messages", 4) 的含义：
       历史一旦超过 300 token，就把除最近 4 条之外的消息全部压缩成摘要
   - 两者差距越大，摘要触发越频繁（压缩掉的内容多，但很快又会超阈值）
   - 建议 keep 对应的 token 量明显小于 trigger，否则会频繁触发摘要浪费调用

7. Checkpointer 与 thread_id 的作用
   - Checkpointer 是 LangGraph 的状态持久化机制，保存 Agent 的完整 state
   - state 中的 messages 就是对话历史，因此「持久化状态」== 「持久化记忆」
   - thread_id 是会话标识，作用等价于第 10 章的 session_id：
       同一个 thread_id 的多次调用共享同一份状态（同一个会话）
       不同 thread_id 的状态完全隔离（不同会话）
   - InMemorySaver 存在内存中，进程退出即丢失；
     生产环境可换 SqliteSaver / PostgresSaver / RedisSaver

8. Checkpointer 与 RunnableWithMessageHistory 的区别
   - RunnableWithMessageHistory（第 10 章）：只持久化「消息历史」，适用于简单链
   - Checkpointer（本章）：持久化 Agent 的「全部状态」，
     包括消息、工具调用中间结果、自定义 state 字段，适用于 Agent 与多步流程
   - Checkpointer 还支持时间旅行（回放到任意历史检查点）与断点续跑

9. 本示例的数据流转全过程
   用户输入 "你好"
     ↓ agent.stream({"messages": [{"role":"user","content":"你好"}]}, config)
     ↓ Checkpointer 按 thread_id 读取已有 state.messages
     ↓ SummarizationMiddleware 检查 token 数是否超过 300
     │    未超过 → 原样放行
     │    已超过 → 调用 llm 把早期消息压缩成摘要，只保留最近 4 条原文
     ↓ 组装 [system_prompt, 摘要消息, ...最近4条..., HumanMessage("你好")]
     ↓ llm 流式生成 AIMessageChunk
     ↓ Checkpointer 把本轮的用户消息与 AI 回复写回 state

10. stream_mode 参数的含义
   - "messages"  逐 token 产出 (消息块, 元数据) 二元组，适合做打字机效果
   - "values"    每步产出完整的 state 快照
   - "updates"   每步只产出 state 的增量变更
   - 本例用 "messages"，因此循环中要解包成 chunk, _ 两个变量

11. 注意事项
   - 摘要会额外消耗一次 LLM 调用，trigger 设置过小会显著增加成本
   - 摘要是有损压缩，关键信息（人名、偏好、结论）可能在多次压缩后丢失
   - 摘要替换是写入 state 的，一旦压缩就不可逆，原文无法恢复
   - InMemorySaver 进程退出即丢失，仅适合 demo 与测试
   - agent.get_state(config) 返回的 messages 是「摘要后」的结果，
     能直接观察到早期消息被替换成摘要消息的现象

12. 对照 0.x ConversationSummaryBufferMemory 的额外坑（历史参考）
   - 0.x 把摘要默认设为 system 角色，消息列表会变成 [system, system, human, ai, ...] 连续多条 system 消息；部分聊天模型（如百度文心）不支持多条 system，且要求历史必须是 1 条 Human + 1 条 AI 的配对格式，直接套用可能报错
   - 0.x 在极端场景（首轮短、次轮长）会执行两次 token 长度计算，若不异步处理对话会变慢
   - 0.x 压缩的源码逻辑（langchain/memory/summary_buffer.py 的 prune）：超过 max_token_limit 时不断 pop 最早的消息，再调用 predict_new_summary 更新 moving_summary_buffer；1.x 的 SummarizationMiddleware 把这套逻辑下沉为 middleware，行为等价但更解耦


In [ ]:
import dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
dotenv.load_dotenv()
llm = ChatOpenAI(model="deepseek-v4-pro")
agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="你是DeepSeek开发的聊天机器人，请根据对应的上下文回复用户问题",
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("tokens", 300),
            keep=("messages", 4),
        ),
    ],
    checkpointer=InMemorySaver(),
)
config = {"configurable": {"thread_id": "muxiaoke"}}
while True:
    query = input("Human: ")
    if query == "q":
        exit(0)
    print("AI: ", flush=True, end="")
    for chunk, _ in agent.stream(
            {"messages": [{"role": "user", "content": query}]},
            config=config,
            stream_mode="messages",
    ):
        print(chunk.text, flush=True, end="")
    print("")
    print("history: ", agent.get_state(config).values["messages"])